In [4]:
import pandas as pd

# Load your clinical notes excel
clinical_df = pd.read_excel(r"C:\Users\cutet\Downloads\UCSD_PTGBM-clinical-information_v3_2026-12-Mar.xlsx", 
                             sheet_name=1)  # sheet index 1 = data sheet, not data dictionary

def safe_get(row, field, default="unknown"):
    val = row.get(field, default)
    if pd.isna(val) or str(val).strip() == "":
        return default
    return str(val).strip()

def synthesize_clinical_note(row):
    
    age        = safe_get(row, "Patient's Age")
    sex        = safe_get(row, "Sex at birth")
    diagnosis  = safe_get(row, "Primary Diagnosis")
    grade      = safe_get(row, "Grade")
    mgmt       = safe_get(row, "MGMT")
    idh        = safe_get(row, "IDH")
    codeletion = safe_get(row, "1p19q")
    atrx       = safe_get(row, "ATRX")

    # WHO 2021 diagnosis takes priority, fall back to Non-WHO
    who = safe_get(row, "WHO 2021 Diagnosis")
    non_who = safe_get(row, "Non WHO 2021 Diagnosis")
    if who != "unknown":
        diagnosis_line = f"WHO 2021 classification: {who}"
    elif non_who != "unknown":
        diagnosis_line = f"Diagnosis (pre-WHO 2021): {non_who}"
    else:
        diagnosis_line = f"Primary diagnosis: {diagnosis}"

    # Surgery
    surgery        = safe_get(row, "Surgery ")
    num_surgeries  = safe_get(row, "Number of surgeries")
    surgery_extent = safe_get(row, "Surgery extend")

    if surgery.lower() == "yes":
        if surgery_extent != "unknown" and num_surgeries != "unknown":
            surgery_line = f"The patient underwent {num_surgeries} surgical procedure(s) with extent of resection: {surgery_extent}."
        elif surgery_extent != "unknown":
            surgery_line = f"The patient underwent surgery with extent of resection: {surgery_extent}."
        elif num_surgeries != "unknown":
            surgery_line = f"The patient underwent {num_surgeries} surgical procedure(s)."
        else:
            surgery_line = "The patient underwent surgery."
    elif surgery.lower() == "no":
        surgery_line = "The patient did not undergo surgery."
    else:
        surgery_line = "Surgical history unknown."

    # Radiation
    radiation         = safe_get(row, "Radiation")
    num_rad_courses   = safe_get(row, "Number of radiation courses")

    if radiation.lower() == "yes":
        if num_rad_courses != "unknown":
            radiation_line = f"Radiation therapy was administered over {num_rad_courses} course(s)."
        else:
            radiation_line = "Radiation therapy was administered."
    elif radiation.lower() == "no":
        radiation_line = "The patient did not receive radiation therapy."
    else:
        radiation_line = "Radiation history unknown."

    # Chemotherapy
    chemo = safe_get(row, "1st Chemo type")
    if chemo != "unknown":
        chemo_line = f"First-line chemotherapy agent: {chemo}."
    else:
        chemo_line = "Chemotherapy history unknown."

    # Molecular markers
    molecular_line = (
        f"Molecular profile: IDH {idh}, MGMT {mgmt}, "
        f"1p19q {codeletion}, ATRX {atrx}."
    )

    # Assemble full note
    note = (
        f"Patient is a {age} year old {sex}. "
        f"{diagnosis_line}, Grade {grade}. "
        f"{molecular_line} "
        f"{surgery_line} "
        f"{radiation_line} "
        f"{chemo_line}"
    )

    return note

# Apply to all rows
clinical_df["synthesized_note"] = clinical_df.apply(synthesize_clinical_note, axis=1)

# Keep only case_id and synthesized note
result_df = clinical_df[["ID", "synthesized_note"]].copy()
result_df.columns = ["case_id", "synthesized_note"]

# Save to excel
output_path = r"C:\Users\cutet\Desktop\ClinIQ\synthesized_clinical_notes.xlsx"
result_df.to_excel(output_path, index=False)

print(f"Saved {len(result_df)} synthesized notes to {output_path}")
print(f"\nSample note:\n{result_df.iloc[0]['synthesized_note']}")

Saved 243 synthesized notes to C:\Users\cutet\Desktop\ClinIQ\synthesized_clinical_notes.xlsx

Sample note:
Patient is a 20 year old Male. Diagnosis (pre-WHO 2021): Glioblastoma, Grade 4. Molecular profile: IDH Wild type, MGMT Methylated, 1p19q Intact, ATRX Loss. The patient underwent 2 surgical procedure(s) with extent of resection: GTR. Radiation therapy was administered over 1.0 course(s). First-line chemotherapy agent: Temodar.


In [ ]:
#Not all cases have scans, we keep only the ones we have the scans of : aka, 184
import pandas as pd
from pathlib import Path

# Load synthesized notes
notes_df = pd.read_excel(r"C:\Users\cutet\Desktop\ClinIQ\synthesized_clinical_notes.xlsx")

# Extract case IDs from image filenames
# UCSD-PTGBM-0002_01_T1post.nii → read from back → before first "_" from back → UCSD-PTGBM-0002_01
image_dir = Path(r"C:\Users\cutet\Downloads\PKG - UCSD-PTGBM-v1\Images")

image_case_ids = set()
for f in image_dir.glob("*.nii"):
    # Split from the right, take everything before the last underscore segment
    # UCSD-PTGBM-0002_01_T1post → rsplit on _ once → ['UCSD-PTGBM-0002_01', 'T1post']
    case_id = f.stem.rsplit("_", 1)[0]
    image_case_ids.add(case_id)

print(f"Unique case IDs found in Images directory: {len(image_case_ids)}")
print(f"Rows in synthesized notes before filtering: {len(notes_df)}")

# Filter notes to only keep rows where case_id exists in images
filtered_df = notes_df[notes_df["case_id"].isin(image_case_ids)]

print(f"Rows after filtering: {len(filtered_df)}")
print(f"Rows removed: {len(notes_df) - len(filtered_df)}")

# Save filtered notes back
output_path = r"C:\Users\cutet\Desktop\ClinIQ\synthesized_clinical_notes.xlsx"
filtered_df.to_excel(output_path, index=False)
print(f"\nSaved filtered notes to {output_path}")